# Plotting Cyclotomic Rings

In this notebook, let us explore the structure and symmetries of cyclotomic rings.

In [ ]:
:dep tilezz = { path = "..", features = ["raster"] }

use std::collections::HashSet;

use tilezz::cyclotomic::*;
use tilezz::cyclotomic::geometry::point_mod_rect;
use tilezz::vis::draw::rainbow;
use tilezz::vis::plotutils::{points_bounds, P64, R64};
use tilezz::vis::scene::{Color, Fill, Item, MarkerShape, Scene, Stroke, Viewport};

## Visualizing Points Reachable In A Fixed Unit Square

The naive approach to compute points in the ring would be doing some simple breadth-first search from the origin, but this is inconvenient if all we want in the end are points within one unit square.

But all starting points are equal. So we can limit our attention to a single unit square and all other neighborhoods will be the same (modulo translation).

So let us explore the points of the ring that we can reach inside the unit square with corners $(0,0)$ and $(1,1)$, i.e. we explore reachable ring elements starting from $0, 1, i$ and $1+i$.

To saturate our set of points, we will use the following algorithm:

* Given a number of rounds $n \in \mathbb{N}$, in each round $i$ go in all possible directions from every point that was reached in round $i-1$ the first time.
* If the point is outside the unit square, normalize it back (by subtracting or adding $1$ or $i$ as often as needed).

Note that the normalized points would have been reached from some adjacent unit square, i.e. some other Gaussian integer, by symmetry.

This way we can populate points inside a single unit square without thinking about the true predecessor points (that can be arbitrarily far away).

The resulting set of points in the unit square is guaranteed to be reachable from *some* $z \in \mathbb{Z}[\zeta_4]$ in at most $n$ steps.

As by assumption we only consider rings that include $\mathbb{Z}[\zeta_4]$ as a subring, all resulting points are valid.

Here is the code for the actual algorithm, the code is more generic so we can do some other experiments later:

In [ ]:
/// Compute the levels of points reachable within the unit square from any Gaussian integer in n steps.
fn explore<ZZ: HasZZ4>(n: usize, mod_unit_square: bool) -> Vec<Vec<ZZ>>
{
    // we start at the corners of the unit square
    let start_pts: &[ZZ] = if mod_unit_square { &[ZZ::zero(), ZZ::one(), ZZ::one_i(), ZZ::one() + ZZ::one_i()] } else {&[ZZ::zero()]};

    let mut visited: HashSet<ZZ> = HashSet::from_iter(start_pts.to_vec().into_iter());
    let mut round_pts: Vec<Vec<ZZ>> = Vec::new();
    round_pts.push(start_pts.to_vec());

    let anchor: ZZ = ZZ::zero(); // unit square anchored at the origin
    // in each round, we go one unit step in every possible direction
    for i in 1..=n {
        let last = round_pts.last().unwrap();
        let mut curr: Vec<ZZ> = Vec::new();
        for p in last.iter() {
            for i in 0..ZZ::turn() {
                let mut p_dest: ZZ = *p + <ZZ as Units>::unit(i);
                if mod_unit_square {
                    // normalize back into unit square (if enabled)
                    p_dest = point_mod_rect(&p_dest, &anchor, (1, 1));
                }
                let p_dest = p_dest;
                
                if !visited.contains(&p_dest) {
                    visited.insert(p_dest);
                    curr.push(p_dest);
                }
            }
        }
        print!("{}{}", curr.len(), if i==n { "" } else {"+"}); // print number of new points
        if i % 10 == 0 {
            println!("")
        }
        round_pts.push(curr);
    }
    let num_pts: usize = round_pts.iter().map(|v| v.len()).sum();
    println!("\n= {num_pts}");
    return round_pts;
}

And here is some utility code for plotting:

In [ ]:
/// Build a Scene containing one round-`level`'s point cloud,
/// shifted by `(dx, dy)` math units. The cell has a light-grey
/// border and an axis crosshair at its origin so subdivisions of
/// the canvas remain easy to read.
fn add_cell(
    scene: &mut Scene,
    points: &[P64],
    bounds: R64,
    (dx, dy): (f64, f64),
    marker_size: f64,
    color: Color,
) {
    let ((mn_x, mn_y), (mx_x, mx_y)) = bounds;
    let w = mx_x - mn_x;
    scene.push(Item::Polygon {
        points: vec![
            (dx + mn_x, dy + mn_y),
            (dx + mx_x, dy + mn_y),
            (dx + mx_x, dy + mx_y),
            (dx + mn_x, dy + mx_y),
        ],
        fill: None,
        stroke: Some(Stroke::solid(Color::rgb(180, 180, 180), 0.005 * w)),
        arrow: None,
    });
    scene.push(Item::Segment {
        a: (dx + mn_x, dy),
        b: (dx + mx_x, dy),
        stroke: Stroke::solid(Color::rgb(225, 225, 225), 0.003 * w),
        arrow: None,
    });
    scene.push(Item::Segment {
        a: (dx, dy + mn_y),
        b: (dx, dy + mx_y),
        stroke: Stroke::solid(Color::rgb(225, 225, 225), 0.003 * w),
        arrow: None,
    });
    let outline = (marker_size * 0.10).max(0.005 * w);
    for &(x, y) in points {
        scene.push(Item::Marker {
            center: (x + dx, y + dy),
            shape: MarkerShape::Circle,
            size: marker_size,
            fill: Some(Fill::solid(color)),
            stroke: Some(Stroke::solid(Color::BLACK, outline)),
        });
    }
}

/// Render a grid of point clouds (one cell per level) into a single
/// SVG-displayable Scene. `levels[i]` is drawn into the `i`-th
/// cell, coloured from a rainbow palette.
fn grid_scene(
    levels: &[&[P64]],
    bounds: R64,
    cols: usize,
    marker_size: f64,
) -> Scene {
    let n = levels.len();
    let rows = n.div_ceil(cols.max(1));
    let ((mn_x, mn_y), (mx_x, mx_y)) = bounds;
    let cell_w = mx_x - mn_x;
    let cell_h = mx_y - mn_y;
    let gap = 0.05 * cell_w;
    let palette: Vec<Color> = rainbow(n.max(1), 1.0, 0.5);
    let mut scene: Scene = Scene::new().with_background(Color::WHITE);
    for (i, pts) in levels.iter().enumerate() {
        let col = i % cols;
        let row = i / cols;
        let dx = col as f64 * (cell_w + gap) - mn_x;
        let dy = (rows - 1 - row) as f64 * (cell_h + gap) - mn_y;
        add_cell(&mut scene, pts, bounds, (dx, dy), marker_size, palette[i]);
    }
    scene
}

/// Convenience: compute a square viewport that fits the result of
/// [`grid_scene`] at `img_w` pixels wide. Padding 16px.
fn grid_viewport(scene: &Scene, img_w: u32) -> Viewport {
    let ((mn_x, mn_y), (mx_x, mx_y)) = scene.auto_bounds().unwrap();
    let math_w = mx_x - mn_x;
    let math_h = mx_y - mn_y;
    let img_h = ((math_h / math_w) * img_w as f64).round().max(1.0) as u32;
    Viewport::rect_for(img_w, img_h, ((mn_x, mn_y), (mx_x, mx_y)), 16)
}

/// Drive `explore` and project each level's points into f64.
fn compute_points<ZZ: HasZZ4>(
    num_rounds: usize,
    mod_unit_square: bool,
) -> Vec<Vec<P64>>
{
    explore::<ZZ>(num_rounds, mod_unit_square)
        .iter()
        .map(|v| v.iter().map(|p| p.xy()).collect())
        .collect()
}

First, let us compute the points in $\mathbb{Z}[\zeta_{12}]$ for some fixed number of rounds:

In [ ]:
let points: Vec<Vec<P64>> = compute_points::<ZZ12>(64, true);
let bounds: R64 = points_bounds(points.iter()).unwrap();

Already here some interesting phenomena are showing up in $\mathbb{Z}[\zeta_{12}]$. Let $i$ be the round and $n_i$ the number of points we discover. Then:

* if $i$ is even, then in round $i+1$ there are also $n_i$ new points
* if in round $i$ we discovered $n_i$ points, in round $i+2$ there will be $n_i + 8$ new points

This number sequence looks like it has a polynomial closed form. The pattern of every next two rounds having the same number of new points is, I guess, due to certain linear dependencies between the directional unit steps. Probably all this can be explained by some well-known mathematical properties.

Now let us plot the points. It's a lot of points, so let us look at different subsets and skip a bunch of exploration steps, to get a clearer picture: *(I recommend using the notebook in full-width mode)*

In [ ]:
// 2x2 panel: each cell shows BFS rounds at a given stride
// (overlaid, so you can see how shells layer up).
let strides: [usize; 4] = [4, 5, 7, 8];
let mut combined: Scene = Scene::new().with_background(Color::WHITE);
let cell_w: f64 = bounds.1.0 - bounds.0.0;
let cell_h: f64 = bounds.1.1 - bounds.0.1;
let gap: f64 = 0.05 * cell_w;
let palette: Vec<Color> = rainbow(points.len().max(1), 1.0, 0.5);
let marker_size: f64 = 0.02 * cell_w;
for (panel_idx, &stride) in strides.iter().enumerate() {
    let col = panel_idx % 2;
    let row = panel_idx / 2;
    let dx = col as f64 * (cell_w + gap) - bounds.0.0;
    let dy = (1 - row) as f64 * (cell_h + gap) - bounds.0.1;
    add_cell(&mut combined, &[], bounds, (dx, dy), marker_size, Color::WHITE);
    for i in (0..points.len()).step_by(stride).rev() {
        let outline = marker_size * 0.10;
        for &(x, y) in &points[i] {
            combined.push(Item::Marker {
                center: (x + dx, y + dy),
                shape: MarkerShape::Circle,
                size: marker_size,
                fill: Some(Fill::solid(palette[i])),
                stroke: Some(Stroke::solid(Color::BLACK, outline)),
            });
        }
    }
}

let vp: Viewport = grid_viewport(&combined, 1200);
combined.display(&vp)


In [ ]:
// 4x4 grid: rounds 1..16 each in its own cell.
let scene: Scene = {
    let levels: Vec<&[P64]> = (1..=16).map(|i| points[i].as_slice()).collect();
    grid_scene(&levels, bounds, 4, 0.02 * (bounds.1.0 - bounds.0.0))
};
let vp: Viewport = grid_viewport(&scene, 1200);
scene.display(&vp)

What we see are the first few of the infinitely many layers of new points obtained by the closure process of $\mathbb{Z}[\zeta_4]$ under the additional degrees of freedom granted by $\mathbb{Z}[\zeta_{12}]$.

Something is happening here that at least *I* would not have expected, and I doubt that this is something you easily expect by just looking at the algebra.

Remember that the unit square is a representative "tile" of the complex plane, which would look everywhere like this if we could have started on *every* Gaussian integer.

The points we found in each round all fall onto two symmetric "rectangles" (in certain rounds they seem to almost merge into one).

This behavior is especially visible for the later rounds, with many points, but if you look closely, this is what happens for *each* round.

Now, that some symmetry shows up is not surprising - rather expected. But:
* Why do we get these rectangular shapes?
* Why do we get a pattern of **two** rectangles, not some different number?

Maybe some number theorist can explain it as a trivial consequence of some property. Or someone will point out a logic flaw in the exploration, or find a bug in the code.

In the meantime, **all I can say is that this is where math and art meet. We only are uncovering what is already there.** Nothing is being created here, only discovered and marvelled at.

### Breadth-First-Exploration from the Origin

For comparison, now let us do the naive exploration starting from the origin. It will reveal different beautiful and interesting patterns for us to look at.

In [ ]:
let points: Vec<Vec<P64>> = compute_points::<ZZ12>(16, false);
let bounds: R64 = points_bounds(points.iter()).unwrap();

In [ ]:
let scene: Scene = {
    let levels: Vec<&[P64]> = (0..16).map(|i| points[i].as_slice()).collect();
    grid_scene(&levels, bounds, 4, 0.02 * (bounds.1.0 - bounds.0.0))
};
let vp: Viewport = grid_viewport(&scene, 1200);
scene.display(&vp)

## Other Visualization Ideas

If you want, you can try to adapt the code to remember for each point from which direction it was discovered (i.e., which unit was added to reach it). This could reveal other interesting patterns in neighborhoods of points.

I have also been thinking about constructing a graph (remembering the edges of the connectivity), but the question is whether there is a nice way to make use of that information visually.

Another direction could be unrolling the discovery rounds into a third dimension as 3D plots — the `vis::scene` model only handles 2D primitives, so a 3D backend would be new work. The `vis::animation` module already produces animated GIFs (see the `cyc_explore` binary), though for inline notebook display we use SVG (no inline GIF rendering in evcxr).

## Bonus: Trying Some Other Rings

As a bonus, let us use the code above to look at the rings $\mathbb{Z}[\zeta_{20}]$ and $\mathbb{Z}[\zeta_{24}]$. The computation takes significantly longer while the result is rather crowded and not as interesting and comprehensible as we have seen in $\mathbb{Z}[\zeta_{12}]$.

But here, the tilezz crate can show off what it can do as a generic cyclotomic ring implementation - to look at other rings, all we have to change is one type argument.

In [ ]:
let points: Vec<Vec<P64>> = compute_points::<ZZ20>(6, true);
let bounds: R64 = points_bounds(points.iter()).unwrap();

In [ ]:
let scene: Scene = {
    let levels: Vec<&[P64]> = (0..6).map(|i| points[i].as_slice()).collect();
    grid_scene(&levels, bounds, 3, 0.02 * (bounds.1.0 - bounds.0.0))
};
let vp: Viewport = grid_viewport(&scene, 1200);
scene.display(&vp)

In [ ]:
let points: Vec<Vec<P64>> = compute_points::<ZZ24>(6, true);
let bounds: R64 = points_bounds(points.iter()).unwrap();

In [ ]:
let scene: Scene = {
    let levels: Vec<&[P64]> = (0..6).map(|i| points[i].as_slice()).collect();
    grid_scene(&levels, bounds, 3, 0.02 * (bounds.1.0 - bounds.0.0))
};
let vp: Viewport = grid_viewport(&scene, 1200);
scene.display(&vp)

In [ ]:
let points: Vec<Vec<P64>> = compute_points::<ZZ20>(6, false);
let bounds: R64 = points_bounds(points.iter()).unwrap();

In [ ]:
let scene: Scene = {
    let levels: Vec<&[P64]> = (0..6).map(|i| points[i].as_slice()).collect();
    grid_scene(&levels, bounds, 3, 0.02 * (bounds.1.0 - bounds.0.0))
};
let vp: Viewport = grid_viewport(&scene, 1200);
scene.display(&vp)

In [ ]:
let points: Vec<Vec<P64>> = compute_points::<ZZ24>(6, false);
let bounds: R64 = points_bounds(points.iter()).unwrap();

In [ ]:
let scene: Scene = {
    let levels: Vec<&[P64]> = (0..6).map(|i| points[i].as_slice()).collect();
    grid_scene(&levels, bounds, 3, 0.02 * (bounds.1.0 - bounds.0.0))
};
let vp: Viewport = grid_viewport(&scene, 1200);
scene.display(&vp)